# Recommendation System

In this notebook, a content-based recommendation system is developed using Spotify audio features. Similar songs are identified by comparing feature vectors using cosine similarity.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

## Load Feature Engineered Dataset

In [2]:
df = pd.read_csv("../data/processed/features_dataset.csv")

## Select Audio Features

These audio characteristics will be used to measure similarity between songs.

In [3]:
FEATURES = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_min",
    "time_signature"
]

## Standardize Features

The selected features are standardized before computing cosine similarity.

In [4]:
scaler = StandardScaler()

feature_matrix = scaler.fit_transform(
    df[FEATURES]
)

## Recommendation Function

This function locates a track in the dataset and returns the most similar songs based on cosine similarity.

In [5]:
def recommend_tracks(track_name, df, feature_matrix, top_n=10):
    matches = df[df["track_name"].str.lower() == track_name.lower()]

    if matches.empty:
        return f"'{track_name}' not found in dataset."

    idx = matches.index[0]

    query_vector = feature_matrix[idx].reshape(1, -1)

    similarities = cosine_similarity(
        query_vector,
        feature_matrix
    )[0]

    similar_indices = similarities.argsort()[::-1]

    similar_indices = [
        i for i in similar_indices
        if i != idx
    ][:top_n]

    results = df.iloc[similar_indices][
        [
            "track_name",
            "artists",
            "track_genre",
            "popularity"
        ]
    ].copy()

    results["similarity_score"] = similarities[
        similar_indices
    ]

    return results.reset_index(drop=True)

## Test the Recommendation System

In [6]:
recommend_tracks(
    "Comedy",
    df,
    feature_matrix,
    top_n=10
)

,track_name,artists,track_genre,popularity,similarity_score
0,JAMAICA,Feid;Sech,pop,5,0.938414
1,空ノムコウ,Maharajan,j-dance,20,0.898946
2,Do My Ladies Run This Party - Single,Cupid,kids,9,0.896783
3,Pull the Catch,DJ Vadim;Fat Freddy's Drop,trip-hop,18,0.892582
4,Scapegoat,Sidhu Moose Wala,hip-hop,58,0.891757
5,Famous,Sidhu Moose Wala;Intense,hip-hop,60,0.890702
6,Bu Aşk Olur Mu,MFÖ,j-rock,33,0.889826
7,Baari Barsi,Salim–Sulaiman;Harshdeep Kaur;Labh Janjua;Amit...,folk,43,0.881795
8,Blessed (feat. Damian Marley),Wizkid;Damian Marley,dancehall,61,0.872840
9,ABC Boo,Super Simple Songs,children,1,0.868072


### Observation

The recommendation system returns songs with similar audio characteristics rather than songs by the same artist. This is a content-based recommendation approach.

## Save Recommendation Assets

In [7]:
joblib.dump(
    scaler,
    "../models/recommender_scaler.pkl"
)

np.save(
    "../models/feature_matrix.npy",
    feature_matrix
)

df.to_csv(
    "../data/processed/app_dataset.csv",
    index=False
)

# Summary

In this notebook:

- Loaded the feature-engineered Spotify dataset.
- Standardized the audio features.
- Built a content-based recommendation system using cosine similarity.
- Tested the recommendation function.
- Saved the scaler, feature matrix, and processed dataset for deployment in the Streamlit application.